# 演習 2 ｜ quantum_half_adder.ipynb

**Toffoli と CNOT で量子半加算器を組む**

---

## このノートブックで確認すること

古典の半加算器を **量子ゲート (Toffoli, CNOT)** で再構築します。

- **入力**: $|a\rangle |b\rangle |0\rangle |0\rangle$
- **出力**: $|a\rangle |b\rangle |\text{Sum}\rangle |\text{Carry}\rangle$
- ただし $\text{Sum} = a \oplus b$, $\text{Carry} = a \land b$

### 真理値表

| $a$ | $b$ | Sum | Carry |
|:-:|:-:|:-:|:-:|
| 0 | 0 | 0 | 0 |
| 0 | 1 | 1 | 0 |
| 1 | 0 | 1 | 0 |
| 1 | 1 | 0 | **1** |

## 所要時間と書く量

- 所要: **5 分**
- **学生が書くのは、セル 3 の TODO (CNOT 2 個 + Toffoli 1 個 = 合計 3 行)**
- セル 1, 2, 4, 5 は「実行するだけ」

---

## なぜ量子で作るのか?

**情工 3 年生なら古典の半加算器は知っている。なぜわざわざ量子で組むのか?**

- 量子ゲートは **ユニタリ = 可逆**。情報を捨てないので、入力 $|a\rangle |b\rangle$ は出力側にも残る
- この半加算器は **Grover のオラクル設計** (次回 #4) で使う基本部品
- **#6 で実機実行** して、古典加算器と比較する予定

---


---
## 環境セットアップ (Google Colab の場合は毎回実行)

第1回ガイダンスの Setup ノートブックと同じ手順です。
ローカル環境 (VSCode 等) で既に Qiskit をインストール済みの場合はスキップしてOK。


In [ ]:
# Qiskit のインストール (1-2 分かかります)
!pip install -q qiskit[visualization] qiskit-aer

# バージョン確認
import qiskit
print(f"Qiskit version: {qiskit.__version__}")
print("インストール完了")

## セル 1 ｜【提供・実行のみ】 セットアップ


In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.quantum_info import Statevector
import numpy as np

SHOTS = 100
sim = AerSimulator()

print("セットアップ完了")


## セル 2 ｜【提供・実行のみ】 qubit の割り当てを理解する

4 つの qubit を次のように割り当てます:

| qubit | 役割 |
|:-:|:-|
| q0 | 入力 $a$ |
| q1 | 入力 $b$ |
| q2 | Sum 桁 (出力、初期値 0) |
| q3 | Carry 桁 (出力、初期値 0) |

### 必要なゲート

- **Sum 桁** ($a \oplus b$): XOR を実現するのは CNOT。$a \oplus b$ を q2 に入れるには、
  - $a$ (q0) を制御、q2 をターゲットとする CNOT
  - $b$ (q1) を制御、q2 をターゲットとする CNOT
  - 2 回 CNOT すると q2 には $a \oplus b$ が残る
- **Carry 桁** ($a \land b$): AND を実現するのは **Toffoli (CCX)**
  - $a$ (q0) と $b$ (q1) を制御、q3 をターゲットとする Toffoli


In [ ]:
# ヘルパ関数: 特定の入力 (a, b) をセットしてから回路を実行し、結果を返す
def run_half_adder(ha_circuit, a, b):
    """半加算器回路を入力 (a, b) で実行する"""
    # 回路に入力セットを前置
    full = QuantumCircuit(4, 4)
    if a == 1:
        full.x(0)    # q0 を |1⟩ に
    if b == 1:
        full.x(1)    # q1 を |1⟩ に
    # q2, q3 は |0⟩ のまま

    # 半加算器回路を合成
    full.compose(ha_circuit, inplace=True)

    # 測定
    full.measure([0, 1, 2, 3], [0, 1, 2, 3])

    counts = sim.run(full, shots=SHOTS).result().get_counts()
    return counts


print("ヘルパ関数を定義しました")


## セル 3 ｜【★ 演習 ★】 半加算器回路を組む

下の `half_adder` を完成させてください (**3 行**):

1. `qc.cx(0, 2)` — $a$ を q2 に XOR
2. `qc.cx(1, 2)` — $b$ を q2 に XOR (q2 には $a \oplus b$ が残る)
3. `qc.ccx(0, 1, 3)` — $a \land b$ を q3 に (Toffoli)

### 予想を書く (30 秒)

- $a = 1, b = 1$ の入力を与えたとき、 (q0, q1, q2, q3) の出力は?
  - あなたの予想: **(___, ___, ___, ___)**


In [ ]:
half_adder = QuantumCircuit(4)

# ─────────────────────────────────────────
# TODO: 半加算器の中身を書く (3 行)
# ヒント:
#   q0 = a, q1 = b, q2 = Sum (初期 0), q3 = Carry (初期 0)
#   Sum = a ⊕ b   → CNOT を 2 回 (a を q2 に、b を q2 に)
#   Carry = a ∧ b → Toffoli を 1 回 (a と b を制御、q3 をターゲット)


# ─────────────────────────────────────────

# 動作確認: a=1, b=1 で実行
counts = run_half_adder(half_adder, a=1, b=1)
print("a=1, b=1 のとき:", counts)
print("→ 期待値: '1100' (q3=1=Carry, q2=0=Sum, q1=1=b, q0=1=a) が出れば正解")
print("  (Qiskit は右から左に q0, q1, q2, q3 の順で並ぶ)")


## セル 4 ｜【提供・実行のみ】 4 入力すべてでテスト

$(a, b)$ の 4 パターンすべてで動作確認します。

Qiskit のビット文字列は **右から** q0, q1, q2, q3 の順。
例: `'1010'` は q3=1, q2=0, q1=1, q0=0 → $a=0, b=1$, Sum=0, Carry=1


In [ ]:
print(f"{'入力 (a,b)':<15} {'結果ビット列':<15} {'意味':<30}")
print("─" * 60)

for (a, b) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    counts = run_half_adder(half_adder, a, b)
    # 最頻値 (ノイズがなければ 1 つだけ出る)
    bitstr = max(counts, key=counts.get)
    # 右から q0, q1, q2, q3
    q0, q1, q2, q3 = bitstr[3], bitstr[2], bitstr[1], bitstr[0]
    meaning = f"a={q0}, b={q1}, Sum={q2}, Carry={q3}"
    print(f"({a}, {b}){'':<9} {bitstr:<15} {meaning}")


## セル 5 ｜【提供・実行のみ】 古典半加算器との比較検証

古典の半加算器と量子版の出力が完全一致することを確認します。


In [ ]:
def classical_half_adder(a, b):
    """古典版の半加算器"""
    sum_bit = a ^ b
    carry_bit = a & b
    return sum_bit, carry_bit


print(f"{'(a,b)':<8} {'古典 Sum,Carry':<18} {'量子 Sum,Carry':<18} {'一致'}")
print("─" * 55)
all_match = True
for (a, b) in [(0, 0), (0, 1), (1, 0), (1, 1)]:
    cs, cc = classical_half_adder(a, b)
    counts = run_half_adder(half_adder, a, b)
    bitstr = max(counts, key=counts.get)
    qs, qc_ = int(bitstr[1]), int(bitstr[0])
    match = "✓" if (cs, cc) == (qs, qc_) else "✗"
    if match == "✗":
        all_match = False
    print(f"({a},{b}){'':<4} ({cs}, {cc}){'':<12} ({qs}, {qc_}){'':<12} {match}")

print()
print("全て一致 ✓" if all_match else "❌ 一致しない入力がある")


## セル 6 ｜【考察】 自分の言葉でまとめる

以下の質問に日本語で答えてください (このセルを編集)。
AI に聞いても構いませんが、**AI の答えを自分の言葉で言い換えてから** 書いてください。

---

### 1. なぜ Sum の計算に CNOT を 2 回使うのか?

「CNOT 1 回で $a \oplus b$ を q2 に入れれば十分」と考えるかもしれないが、実際は 2 回必要。なぜ?

**ヒント**: CNOT は `q[target] = q[target] ⊕ q[control]` で、q2 の初期値は 0。

**あなたの説明**: _______________________________________

---

### 2. Toffoli が「3 入力 AND の量子版」と呼ばれる理由

q3 を 0 で初期化したとき、Toffoli(q0, q1, q3) の出力は何か?
真理値表から確認して説明してください。

**あなたの説明**: _______________________________________

---

### 3. 全加算器に拡張するには?

全加算器は入力 $a, b, c_{\text{in}}$ に対して、
- $\text{Sum} = a \oplus b \oplus c_{\text{in}}$
- $\text{Carry} = (a \land b) \lor ((a \oplus b) \land c_{\text{in}})$

を計算する。これを量子で組むには、どんなゲートが何個必要か?
(ヒント: 半加算器 2 つ + OR で組める。OR は古典的には単純だが、量子では少し工夫がいる)

**あなたの答え**: _______________________________________

---

### 4. 今日の学び (自由記述)

古典の論理回路が知識としてあれば、量子の算術回路は「古典と同じ発想 + 可逆性」で組めることを体験した。
この発想の射程について感じたことがあれば書いてください。

**あなたの感想**: _______________________________________

---


## 📌 段階的ヒント (詰まったら開く)

### 半加算器のヒント

**L1** (ゆるいヒント):
Sum は XOR、Carry は AND。XOR は CNOT、AND は Toffoli で実装できる。
qubit 割り当ては: q0 = a, q1 = b, q2 = Sum (初期 0), q3 = Carry (初期 0)。

**L2** (構造のヒント):
- Sum = a ⊕ b: q0 を制御、q2 をターゲットとする CNOT で q2 に a を書き込む。次に q1 を制御、q2 をターゲットとする CNOT で q2 に b を XOR する。結果、q2 = 0 ⊕ a ⊕ b = a ⊕ b
- Carry = a ∧ b: q0 と q1 を制御、q3 をターゲットとする Toffoli で q3 に a ∧ b を書き込む

**L3** (ほぼ答え):
```python
half_adder.cx(0, 2)
half_adder.cx(1, 2)
half_adder.ccx(0, 1, 3)
```

---


## 📚 持ち帰り課題 (任意、所要 20-30 分)

### 課題 A: 全加算器の実装

入力 $a, b, c_{\text{in}}$ から Sum と Carry$_{\text{out}}$ を計算:

- $\text{Sum} = a \oplus b \oplus c_{\text{in}}$
- $\text{Carry}_{\text{out}} = (a \land b) \lor ((a \oplus b) \land c_{\text{in}})$

### 課題 B: 2-bit 加算器

$a = a_1 a_0$, $b = b_1 b_0$ を入力として、$a + b$ の 3 ビット結果を計算。
半加算器と全加算器を組み合わせる。

### 課題 C: 2 の補数を使った減算器

$a - b$ を計算する回路。負の結果は 2 の補数表現で出す。

### 雛形 (課題 A)


In [ ]:
# 全加算器の雛形

full_adder = QuantumCircuit(5)  # q0=a, q1=b, q2=c_in, q3=Sum, q4=Carry_out

# TODO: 実装する
# ヒント: 半加算器 2 つ + OR で組める
#   半加算器 1: a, b → Sum1, Carry1
#   半加算器 2: Sum1, c_in → Sum_final, Carry2
#   OR: Carry1 OR Carry2 → Carry_out
#
# 量子での OR:
#   OR(x, y) = NOT(NOT(x) AND NOT(y))
#   つまり X を各入力にかけてから Toffoli、その後 出力にX (入力はX で戻す)
#   または: a ⊕ b ⊕ (a AND b) でも OR になる (AND の結果を知っていれば)


# 実装後のテスト用ヘルパ (コメントアウトを外して使う)
# def run_full_adder(fa, a, b, cin):
#     full = QuantumCircuit(5, 5)
#     if a == 1: full.x(0)
#     if b == 1: full.x(1)
#     if cin == 1: full.x(2)
#     full.compose(fa, inplace=True)
#     full.measure(range(5), range(5))
#     counts = sim.run(full, shots=SHOTS).result().get_counts()
#     bitstr = max(counts, key=counts.get)
#     return int(bitstr[1]), int(bitstr[0])   # Sum, Carry_out

# for (a, b, cin) in [(0,0,0), (0,0,1), (1,1,0), (1,1,1)]:
#     s, c = run_full_adder(full_adder, a, b, cin)
#     print(f"({a},{b},{cin}) → Sum={s}, Carry_out={c}")


### 条件

- **AI 活用可**: ChatGPT, Claude, Copilot など
- ただし **最終的に、自分の言葉で回路の動作を説明できる** こと
- 課題 A は必須、課題 B / C は任意
- 実行結果のスクリーンショット + 2-3 行の考察を提出

---

### 次回予告

- #4 (来週) では、ここで作った半加算器の発想を使って **ライツアウトパズルのオラクル** を設計します
- 「マスを押す」操作 = CNOT の連鎖、「全消灯判定」 = Toffoli を組み合わせた判定回路
- 今日の道具 (位相キックバック + Toffoli + 半加算器の発想) が全部必要になります
- #6 では、ここで作った半加算器を **実機で実行** して、古典との差を見る予定
